# Private phase 2: fine-tune doc lap toan bo test pose

Moi test pose load lai checkpoint PLY 30k cua scene, cham tat ca train view,
chon top-25 va fine-tune 3.000 optimizer update. C2F va pose-aware sampling
deu tat trong local stage; densify/split chi chay 1.500 step dau.

Notebook cho phep chon mot/nhieu scene, chay toan bo pose cua moi scene trong
mot lan, render PNG va dong goi ZIP dung cau truc `scene/image.png`.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

WORK_ROOT = Path('/kaggle/working')
REPO_DIR = WORK_ROOT / 'Improved-GS'
REPO_BRANCH = 'agent/test-pose-finetune'

# ========== CHI SUA CAC BIEN TRONG KHOI NAY ==========
# De trong de tu tim data phase 2 da attach vao Kaggle.
PHASE2_DIR_OVERRIDE = ''

# Mot scene, nhieu scene, hoac [] de chay tat ca 7 scene.
SCENE_NAMES = ['HCM0421']

# Chi can khai bao scene co nhieu checkpoint 30k tren Kaggle input.
BASE_MODEL_OVERRIDES = {
    # 'HCM0421': '/kaggle/input/model-dataset/vai_models/HCM0421',
}

# Neu khong khai bao, notebook doc budget tu training_parameters.json cua model.
BUDGET_OVERRIDES = {
    # 'HCM0421': 5_500_000,
}
FALLBACK_BUDGET = 5_500_000

BASE_ITERATION = 30_000
FINE_TUNE_STEPS = 3_000
SPLIT_UNTIL_STEP = 1_500
TOP_K = 25
SIGMA_MULTIPLIER = 3.0
DENSIFY_GRAD_THRESHOLD = 0.0002
SAVE_POSE_MODELS = False
# ======================================================

EXPECTED_PHASE2_SCENES = {
    'bonsai', 'chair', 'HCM0421', 'HCM0539', 'HCM0540', 'HCM0644', 'HCM0674',
}

def scene_names_in(root):
    return {
        path.name for path in root.iterdir()
        if path.is_dir()
        and (path / 'train' / 'images').is_dir()
        and (path / 'train' / 'sparse' / '0' / 'cameras.bin').is_file()
        and (path / 'test' / 'test_poses.csv').is_file()
    }

if PHASE2_DIR_OVERRIDE:
    PHASE2_DIR = Path(PHASE2_DIR_OVERRIDE)
else:
    candidate_roots = sorted({
        pose_path.parents[2]
        for pose_path in Path('/kaggle/input').glob('**/test/test_poses.csv')
        if len(pose_path.parents) >= 3
    })
    phase2_candidates = [
        root for root in candidate_roots
        if root.is_dir() and EXPECTED_PHASE2_SCENES <= scene_names_in(root)
    ]
    if len(phase2_candidates) != 1:
        raise RuntimeError(
            'Khong tu xac dinh duoc duy nhat data phase 2. '
            f'Tim duoc: {phase2_candidates}. Hay dat PHASE2_DIR_OVERRIDE.'
        )
    PHASE2_DIR = phase2_candidates[0]

if not PHASE2_DIR.is_dir():
    raise FileNotFoundError(PHASE2_DIR)
discovered_scenes = scene_names_in(PHASE2_DIR)
if not EXPECTED_PHASE2_SCENES <= discovered_scenes:
    raise ValueError(
        f'Data phase 2 thieu scene: {sorted(EXPECTED_PHASE2_SCENES - discovered_scenes)}'
    )
AVAILABLE_SCENES = sorted(EXPECTED_PHASE2_SCENES)
if len(set(SCENE_NAMES)) != len(SCENE_NAMES):
    raise ValueError(f'SCENE_NAMES bi trung: {SCENE_NAMES}')
missing_scenes = sorted(set(SCENE_NAMES) - set(AVAILABLE_SCENES))
if missing_scenes:
    raise ValueError(f'Khong tim thay scene: {missing_scenes}')
SELECTED_SCENES = list(SCENE_NAMES) if SCENE_NAMES else AVAILABLE_SCENES

DATA_ROOT = WORK_ROOT / 'vai_phase2_cleaned'
MODEL_ROOT = WORK_ROOT / 'vai_phase2_pose_models'
PNG_ROOT = WORK_ROOT / 'vai_phase2_pose_png'
selection_tag = 'all' if SELECTED_SCENES == AVAILABLE_SCENES else '_'.join(SELECTED_SCENES)
ZIP_PATH = WORK_ROOT / f'private_phase2_pose_finetune_{selection_tag}_png.zip'

print('Phase 2 data:', PHASE2_DIR)
print('Available scenes:', AVAILABLE_SCENES)
print('Selected scenes:', SELECTED_SCENES)
print('PNG ZIP:', ZIP_PATH)

In [ ]:
# Clone hoac cap nhat dung nhanh fine-tune.
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--recursive', '--branch', REPO_BRANCH,
        'https://github.com/mdd206/Improved-GS.git', str(REPO_DIR),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run([
        'git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH,
    ], check=True)
os.chdir(REPO_DIR)
assert (REPO_DIR / 'vai_test_pose_finetune.py').is_file(), REPO_DIR
print('Repo:', Path.cwd())

In [ ]:
# Cai dependency Python, COLMAP neu co scene radial, va CUDA extension.
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'numpy==1.26.1', 'opencv-python==4.10.0.82',
    'setuptools==69.5.1', 'ninja', 'tqdm', 'plyfile',
], check=True)

from vai.colmap_io import read_intrinsics_binary

RAW_CAMERA_MODELS = {}
for scene_name in SELECTED_SCENES:
    cameras = read_intrinsics_binary(
        str(PHASE2_DIR / scene_name / 'train' / 'sparse' / '0' / 'cameras.bin')
    )
    if len(cameras) != 1:
        raise ValueError(f'{scene_name} phai co dung 1 camera, nhan {len(cameras)}')
    RAW_CAMERA_MODELS[scene_name] = next(iter(cameras.values())).model
unsupported_models = sorted({
    model for model in RAW_CAMERA_MODELS.values()
    if model not in {'SIMPLE_RADIAL', 'SIMPLE_PINHOLE'}
})
if unsupported_models:
    raise ValueError(f'Camera model chua ho tro: {unsupported_models}')

def install_colmap_with_apt():
    apt_env = os.environ.copy()
    apt_env['DEBIAN_FRONTEND'] = 'noninteractive'
    options = [
        '-o', 'Dpkg::Use-Pty=0',
        '-o', 'Dpkg::Lock::Timeout=120',
        '-o', 'Acquire::Retries=3',
    ]
    print('Bat dau apt-get update...', flush=True)
    update_result = subprocess.run(
        ['apt-get', *options, 'update'], check=False, env=apt_env,
    )
    if update_result.returncode != 0:
        return False
    install_result = subprocess.run([
        'apt-get', *options, 'install', '-y', '--no-install-recommends', 'colmap',
    ], check=False, env=apt_env)
    return install_result.returncode == 0

def install_colmap_with_conda():
    conda_cli = next(
        (name for name in ('micromamba', 'mamba', 'conda') if shutil.which(name)),
        None,
    )
    if conda_cli is None:
        return False
    colmap_env = WORK_ROOT / 'colmap-env'
    action = 'install' if (colmap_env / 'conda-meta').is_dir() else 'create'
    result = subprocess.run([
        conda_cli, action, '-y', '-p', str(colmap_env), '-c', 'conda-forge', 'colmap',
    ], check=False)
    colmap_bin = colmap_env / 'bin' / 'colmap'
    if result.returncode == 0 and colmap_bin.is_file():
        os.environ['PATH'] = str(colmap_bin.parent) + os.pathsep + os.environ.get('PATH', '')
        return True
    return False

needs_colmap = any(model == 'SIMPLE_RADIAL' for model in RAW_CAMERA_MODELS.values())
if needs_colmap and shutil.which('colmap') is None and not install_colmap_with_apt():
    print('APT khong cai duoc COLMAP, thu Conda fallback.', flush=True)
if needs_colmap and shutil.which('colmap') is None:
    install_colmap_with_conda()
if needs_colmap and shutil.which('colmap') is None:
    raise RuntimeError('Khong the cai COLMAP cho scene SIMPLE_RADIAL')
print('Raw camera models:', RAW_CAMERA_MODELS)
print('COLMAP:', shutil.which('colmap') if needs_colmap else 'khong can')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Hay bat GPU Accelerator tren Kaggle')
cuda_major, cuda_minor = torch.cuda.get_device_capability()
build_env = os.environ.copy()
build_env['MAX_JOBS'] = '2'
build_env['TORCH_CUDA_ARCH_LIST'] = f'{cuda_major}.{cuda_minor}'
for package_dir in [
    'submodules/diff-gaussian-rasterization',
    'submodules/simple-knn',
    'submodules/fused-ssim',
]:
    print('Bat dau build:', package_dir, flush=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-v',
        '--no-build-isolation', '--no-deps', package_dir,
    ], check=True, env=build_env, timeout=1200)
    print('Build xong:', package_dir, flush=True)
print('Done', flush=True)

In [ ]:
# Radial: undistort sang PINHOLE RGBA. Simple pinhole: giu nguyen RGB.
subprocess.run([
    sys.executable, '-u', 'vai_preprocess.py',
    '--input', str(PHASE2_DIR),
    '--output', str(DATA_ROOT),
    '--subset', *SELECTED_SCENES,
    '--overwrite',
], check=True)
print('Preprocess xong:', DATA_ROOT)

In [ ]:
# Tim model 30k, doc budget va chan nham D3/native SIMPLE_RADIAL.
from vai.common import read_pose_rows

def read_ply_vertex_count(ply_path):
    vertex_count = None
    with open(ply_path, 'rb') as handle:
        for _ in range(1000):
            line = handle.readline()
            if not line:
                break
            text = line.decode('ascii', errors='strict').strip()
            if text.startswith('element vertex '):
                vertex_count = int(text.split()[-1])
            if text == 'end_header':
                break
    if vertex_count is None:
        raise ValueError(f'Khong doc duoc vertex count tu {ply_path}')
    return vertex_count

def resolve_base_model(scene_name):
    override = str(BASE_MODEL_OVERRIDES.get(scene_name, '')).strip()
    if override:
        return Path(override)
    ply_candidates = sorted(
        path for path in Path('/kaggle/input').glob(
            f'**/{scene_name}/point_cloud/iteration_{BASE_ITERATION}/point_cloud.ply'
        )
        if path.is_file()
    )
    model_candidates = sorted({path.parents[2] for path in ply_candidates})
    if len(model_candidates) != 1:
        raise RuntimeError(
            f'{scene_name}: can dung 1 model 30k, tim duoc {model_candidates}. '
            'Hay dat BASE_MODEL_OVERRIDES.'
        )
    return model_candidates[0]

BASE_MODELS = {}
SCENE_BUDGETS = {}
MODEL_SUMMARY = {}
for scene_name in SELECTED_SCENES:
    base_model = resolve_base_model(scene_name)
    base_ply = base_model / 'point_cloud' / f'iteration_{BASE_ITERATION}' / 'point_cloud.ply'
    if not base_ply.is_file():
        raise FileNotFoundError(base_ply)
    if 'native_simple_radial' in str(base_model).lower():
        raise ValueError(f'{scene_name}: tu choi D3 model {base_model}')

    parameters_path = base_model / 'training_parameters.json'
    parameters = json.loads(parameters_path.read_text()) if parameters_path.is_file() else {}
    method = str(parameters.get('training_method', '')).lower()
    if method and method != 'improvedgs':
        raise ValueError(f'{scene_name}: base model khong phai ImprovedGS: {method}')
    for disabled_flag in ('coarse_to_fine', 'pose_aware_sampling'):
        if parameters.get(disabled_flag) is True:
            print(
                f'{scene_name}: base tung dung {disabled_flag}; '
                'co che nay se bi tat trong local fine-tune.'
            )

    camera_json_path = base_model / 'cameras.json'
    if camera_json_path.is_file():
        camera_rows = json.loads(camera_json_path.read_text())
        camera_models = {
            str(row.get('camera_model', '')).upper() for row in camera_rows
        }
        if 'SIMPLE_RADIAL' in camera_models:
            raise ValueError(f'{scene_name}: tu choi checkpoint native SIMPLE_RADIAL')

    gaussian_count = read_ply_vertex_count(base_ply)
    budget = int(BUDGET_OVERRIDES.get(
        scene_name,
        parameters.get('budget', FALLBACK_BUDGET),
    ))
    if budget < gaussian_count:
        raise ValueError(
            f'{scene_name}: budget {budget} < Gaussian hien co {gaussian_count}'
        )
    BASE_MODELS[scene_name] = base_model
    SCENE_BUDGETS[scene_name] = budget
    MODEL_SUMMARY[scene_name] = {
        'base_model': str(base_model),
        'gaussian_count': gaussian_count,
        'budget': budget,
        'raw_camera_model': RAW_CAMERA_MODELS[scene_name],
        'test_pose_count': len(read_pose_rows(
            PHASE2_DIR / scene_name / 'test' / 'test_poses.csv'
        )),
    }

print(json.dumps(MODEL_SUMMARY, indent=2))

In [ ]:
# Moi scene chay toan bo test pose: pose_count=-1, render truc tiep PNG.
env = os.environ.copy()
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
RUN_SUMMARY = {}
for scene_name in SELECTED_SCENES:
    source_path = DATA_ROOT / scene_name
    model_path = MODEL_ROOT / scene_name
    command = [
        sys.executable, '-u', 'vai_test_pose_finetune.py',
        '--source_path', str(source_path),
        '--base_model_path', str(BASE_MODELS[scene_name]),
        '--base_iteration', str(BASE_ITERATION),
        '--model_path', str(model_path),
        '--scene_name', scene_name,
        '--pose_start_index', '0',
        '--pose_count', '-1',
        '--fine_tune_steps', str(FINE_TUNE_STEPS),
        '--split_from_step', '0',
        '--split_until_step', str(SPLIT_UNTIL_STEP),
        '--top_k', str(TOP_K),
        '--sigma_multiplier', str(SIGMA_MULTIPLIER),
        '--save_pose_models', str(SAVE_POSE_MODELS).lower(),
        '--training_method', 'improvedgs',
        '--use_las', 'true',
        '--use_eas', 'true',
        '--use_rap', 'true',
        '--use_mu', 'true',
        '--coarse_to_fine', 'false',
        '--pose_aware_sampling', 'false',
        '--densify_grad_threshold', str(DENSIFY_GRAD_THRESHOLD),
        '--budget', str(SCENE_BUDGETS[scene_name]),
        '--resolution', '-1',
        '--data_device', 'cpu',
        '--eval', 'false',
        '--train_test_exp', 'false',
        '--output_root', str(PNG_ROOT),
        '--output_extension', 'png',
        '--save_png', 'false',
        '--evaluate', 'false',
        '--require_gt', 'false',
        '--overwrite', 'true',
        '--progress_bar_width', '100',
    ]
    print('\nBat dau scene:', scene_name)
    print(' '.join(command))
    subprocess.run(command, cwd=REPO_DIR, check=True, env=env)

    manifest_path = model_path / 'test_pose_finetune_manifest.json'
    manifest = json.loads(manifest_path.read_text())
    expected_count = MODEL_SUMMARY[scene_name]['test_pose_count']
    assert manifest['pose_start_index'] == 0
    assert manifest['pose_end_index_exclusive'] == expected_count
    assert manifest['completed_pose_count'] == expected_count
    assert manifest['pose_indices'] == list(range(expected_count))
    assert all(pose['optimizer_updates'] == FINE_TUNE_STEPS for pose in manifest['poses'])
    assert all(pose['selection']['selected_top_k'] == TOP_K for pose in manifest['poses'])
    RUN_SUMMARY[scene_name] = {
        'poses': expected_count,
        'training_seconds': manifest['total_training_seconds'],
        'render_dir': manifest['render_dir'],
    }
    print('Hoan tat scene:', scene_name, RUN_SUMMARY[scene_name])

print(json.dumps(RUN_SUMMARY, indent=2))

In [ ]:
# Validate toan bo PNG cua scene da chon va dong goi mot ZIP.
subprocess.run([
    sys.executable, 'vai_package.py',
    '--phase_dir', str(PHASE2_DIR.parent),
    '--set_name', PHASE2_DIR.name,
    '--submission_dir', str(PNG_ROOT),
    '--zip_path', str(ZIP_PATH),
    '--subset', *SELECTED_SCENES,
    '--output_extension', 'png',
    '--pose_start_index', '0',
    '--pose_count', '-1',
], check=True)

expected_total = sum(MODEL_SUMMARY[name]['test_pose_count'] for name in SELECTED_SCENES)
with zipfile.ZipFile(ZIP_PATH) as archive:
    names = archive.namelist()
    assert len(names) == expected_total
    assert all(name.endswith('.png') for name in names)
    for scene_name in SELECTED_SCENES:
        scene_count = sum(name.startswith(f'{scene_name}/') for name in names)
        assert scene_count == MODEL_SUMMARY[scene_name]['test_pose_count']

print('ZIP san sang:', ZIP_PATH)
print('Scenes:', SELECTED_SCENES)
print('PNG count:', expected_total)
print('ZIP size MB:', round(ZIP_PATH.stat().st_size / (1024 ** 2), 2))